# **A Mini-Project to Clean a Messy Student Survey data**

**The imports**

In [1]:
import pandas as pd
import numpy as np
import io
import re # regex (pattern matching) for cleaning messy text like "16 years"

In [2]:
# 2. Load the deliberately messy CSV dataset (20 rows, 10 columns)
csv_data = """student_id,age,grade_level,gender,device_type,internet_quality,satisfaction_rating,hours_online_learning,preferred_learning,feedback
1,16,10,Female,Laptop,Good,8,5,Video,"Great experience, learned a lot"
2,seventeen,11,M,tablet,excellent,9,6,Interactive,"Good but need better audio"
3,16 years,10,FEMALE,laptop,Poor,3,2,reading,""
4,999,12,Male,Desktop,good,7,4,video,"Okay experience"
5,15,9,f,Phone,Bad,2,1,interactive,"Very difficult with poor internet"
6,16,10,Female,Laptop,Good,8,5,Video,"Great experience, learned a lot"
7,18,12,male,LAPTOP,Excellent,15,8,Mixed,"Excellent program!"
8,,11,Female,Tablet,Fair,6,3,Reading,"Missing some features"
9,14,9,M,laptop,good,5,4,video,""
10,16,10,f,Desktop,Poor,4,3,Interactive,"Need better support"
11,17,11,Male,Phone,excellent,8,6,mixed,"Good overall"
12,-5,8,Female,Laptop,Good,1,0,Video,"Too young for this"
13,16,10,MALE,tablet,Bad,3,2,reading,"Hard to focus"
14,150,12,F,Desktop,Fair,7,5,Interactive,"Interesting but long"
15,sixteen,10,female,Laptop,Good,9,7,Video,"Loved it!"
16,17,11,,Tablet,excellent,6,4,Reading,"Good content"
17,15,9,Male,laptop,poor,2,1,video,"Technical issues"
18,16,10,Female,Desktop,Good,8,5,Mixed,"Great experience"
19,18,12,m,Phone,Excellent,7,6,interactive,"Good but challenging"
20,16,10,FEMALE,Laptop,good,20,5,Video,"Best program ever!\""""

In [3]:
df = pd.read_csv(io.StringIO(csv_data))
print("Loaded:", df.shape)
df.head()

Loaded: (20, 10)


,student_id,age,grade_level,gender,device_type,internet_quality,satisfaction_rating,hours_online_learning,preferred_learning,feedback
0,1,16,10,Female,Laptop,Good,8,5,Video,"Great experience, learned a lot"
1,2,seventeen,11,M,tablet,excellent,9,6,Interactive,Good but need better audio
2,3,16 years,10,FEMALE,laptop,Poor,3,2,reading,NaN
3,4,999,12,Male,Desktop,good,7,4,video,Okay experience
4,5,15,9,f,Phone,Bad,2,1,interactive,Very difficult with poor internet


In [4]:
df.describe()

,student_id,grade_level,satisfaction_rating,hours_online_learning
count,20.00000,20.000000,20.000000,20.000000
mean,10.50000,10.350000,6.900000,4.100000
std,5.91608,1.136708,4.459172,2.125039
min,1.00000,8.000000,1.000000,0.000000
25%,5.75000,10.000000,3.750000,2.750000
50%,10.50000,10.000000,7.000000,4.500000
75%,15.25000,11.000000,8.000000,5.250000
max,20.00000,12.000000,20.000000,8.000000


**M1) Remove duplicate submissions**

In [5]:
# If a student accidentally submitted the survey twice, we keep only the first record.
df = df.drop_duplicates(subset="student_id")
print("Removed duplicates:", df.shape)
print(df["age"])


Removed duplicates: (20, 10)
0            16
1     seventeen
2      16 years
3           999
4            15
5            16
6            18
7           NaN
8            14
9            16
10           17
11           -5
12           16
13          150
14      sixteen
15           17
16           15
17           16
18           18
19           16
Name: age, dtype: object


**M2) Make age numeric**

In [21]:
# The "age" column is messy: values like "sixteen", "16 years", or even invalid numbers like 999.

# Step 1: Convert everything to lowercase strings
ages = df["age"].astype(str).str.lower().str.strip()

# Step 2: replace written words with digit strings
word_to_num = {
    "fourteen": "14",
    "fifteen": "15",
    "sixteen": "16",
    "seventeen": "17",
    "eighteen": "18"
}
ages = ages.replace(word_to_num)

# Step 3: FIXED - remove non-digit characters but keep decimal points
ages = ages.apply(lambda x: re.sub(r"[^0-9.]", "", str(x)))

# Step 4: convert to numeric (invalid strings -> NaN)
df["age"] = pd.to_numeric(ages, errors="coerce")

# Step 5: ADD THIS - handle unrealistic ages
df["age"] = df["age"].apply(lambda x: x if pd.isna(x) or (5< x <= 120) else np.nan)

# Then fill missing ages with the median within the same grade_level group.
df["age"] = df.groupby("grade_level")["age"].transform(lambda x: x.fillna(x.median()))

print(df["age"])

0     16.0
1     17.0
2     16.0
3     18.0
4     15.0
5     16.0
6     18.0
7     17.0
8     14.0
9     16.0
10    17.0
11     NaN
12    16.0
13    18.0
14    16.0
15    17.0
16    15.0
17    16.0
18    18.0
19    16.0
Name: age, dtype: float64


**M4) Standardize gender categories**

In [10]:
print("Gender counts:\n", df["gender"].value_counts(dropna=False), "\n")


Gender counts:
 gender
Female    5
Male      3
M         2
FEMALE    2
f         2
male      1
MALE      1
F         1
female    1
NaN       1
m         1
Name: count, dtype: int64 



In [12]:
# Gender column has values like "M", "male", "FEMALE", "f".
# We standardize them to "Male" or "Female".
gmap = {"m": "Male", "male": "Male", "f": "Female", "female": "Female"}
df["gender"] = df["gender"].astype(str).str.strip().str.lower().map(gmap)
print("Gender counts:\n", df["gender"].value_counts(dropna=False), "\n")
print(df["gender"])

Gender counts:
 gender
Female    11
Male       8
NaN        1
Name: count, dtype: int64 

0     Female
1       Male
2     Female
3       Male
4     Female
5     Female
6       Male
7     Female
8       Male
9     Female
10      Male
11    Female
12      Male
13    Female
14    Female
15       NaN
16      Male
17    Female
18      Male
19    Female
Name: gender, dtype: object


**M5) Force ratings onto a 1–10 scale**

In [14]:
# Convert to numeric, force onto [1, 10].
df["satisfaction_rating"] = pd.to_numeric(df["satisfaction_rating"], errors="coerce").clip(1, 10)
print("Ratings summary:\n", df["satisfaction_rating"].describe(), "\n")

Ratings summary:
 count    20.000000
mean      6.150000
std       2.796144
min       1.000000
25%       3.750000
50%       7.000000
75%       8.000000
max      10.000000
Name: satisfaction_rating, dtype: float64 



**M6) Tidy text columns**

In [15]:
# Clean device_type, internet_quality, preferred_learning, feedback:
for c in ["device_type", "internet_quality", "preferred_learning", "feedback"]:
    s = df[c].astype(str).str.strip()
    s = s.mask(s.eq("") | s.eq("nan"))       # empty strings → NaN
    df[c] = s.where(s.isna(), s.str.title()) # TitleCase if not missing
print("Missing values per column:\n", df.isna().sum(), "\n")

Missing values per column:
 student_id               0
age                      1
grade_level              0
gender                   1
device_type              0
internet_quality         0
satisfaction_rating      0
hours_online_learning    0
preferred_learning       0
feedback                 2
dtype: int64 



**M7) Final check**

In [16]:
# Confirm types are correct and data is reasonable.
df.info()
print(df.head(), "\n")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   student_id             20 non-null     int64  
 1   age                    19 non-null     float64
 2   grade_level            20 non-null     int64  
 3   gender                 19 non-null     object 
 4   device_type            20 non-null     object 
 5   internet_quality       20 non-null     object 
 6   satisfaction_rating    20 non-null     int64  
 7   hours_online_learning  20 non-null     int64  
 8   preferred_learning     20 non-null     object 
 9   feedback               18 non-null     object 
dtypes: float64(1), int64(4), object(5)
memory usage: 1.7+ KB
   student_id   age  grade_level  gender device_type internet_quality  \
0           1  16.0           10  Female      Laptop             Good   
1           2  17.0           11    Male      Tablet        Excel

In [17]:
# Question: What is the median age for students in grade_level == 10 (after cleaning)?
value_to_submit = round(df.loc[df["grade_level"] == 10, "age"].median(), 2)
print("Value to submit:", value_to_submit)

Value to submit: 16.0
